# Phase 2 — FLAN-T5 Translation Fine-tuning
**Continuous Sign Language Translation** · Phase 2 of 2

Loads the frozen `MultiStreamSemanticEncoder` from Phase 1, then fine-tunes a `SignToTextModel` (encoder → attention pooling → adapter → FLAN-T5-small) for utterance-level sign-to-text translation.

> ⚠️ **Run Phase 1 first** — this notebook expects a Phase 1 checkpoint.

**Runtime:** GPU (T4 or better)


In [1]:
# ── Clone repository and install dependencies ──────────────────────────
!git clone https://github.com/bencejdanko/cslt-flan-t5-small-autoencoder.git cslt
%cd cslt
!git pull origin main
!pip install -q -r requirements.txt

Cloning into 'cslt'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 51 (delta 16), reused 41 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 88.89 KiB | 5.55 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/cslt
From https://github.com/bencejdanko/cslt-flan-t5-small-autoencoder
 * branch            main       -> FETCH_HEAD
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.7 MB/s eta 0:00:00


In [2]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


In [4]:
# ── Hugging Face authentication ───────────────────────────────────────
# Set Colab Secret "HF_TOKEN" or paste when prompted. huggingface_hub uses os.environ["HF_TOKEN"]; do not hardcode tokens.
import os
try:
    from google.colab import userdata
    _hf = userdata.get("HF_TOKEN")
    if _hf:
        os.environ["HF_TOKEN"] = _hf
        print("HF_TOKEN loaded from Colab Secrets ✓")
except Exception:
    pass
if not os.environ.get("HF_TOKEN"):
    import getpass
    os.environ["HF_TOKEN"] = getpass.getpass("HF token (hidden): ")


HF_TOKEN loaded from Colab Secrets ✓


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Download Phase 1 checkpoint (if not local)
If you trained Phase 1 in a previous Colab session, download the encoder weights from HuggingFace.

In [5]:
import os
from huggingface_hub import hf_hub_download

PHASE1_CKPT = "/content/phase1_ckpt"

# Download from HuggingFace if no local checkpoint exists
if not os.path.exists(os.path.join(PHASE1_CKPT, "best", "encoder.pt")):
    os.makedirs(os.path.join(PHASE1_CKPT, "best"), exist_ok=True)
    try:
        encoder_path = hf_hub_download(
            repo_id="mycyrilgoud/data255",
            filename="best/encoder.pt",
            local_dir=PHASE1_CKPT,
        )
        print(f"Encoder downloaded to {encoder_path} ✓")
    except Exception as e:
        print(f"Could not download Phase 1 checkpoint: {e}")
        print("Phase 2 will train encoder from scratch.")
else:
    print(f"Phase 1 checkpoint found at {PHASE1_CKPT}/best/ ✓")

best/encoder.pt:   0%|          | 0.00/35.5M [00:00<?, ?B/s]

Encoder downloaded to /content/phase1_ckpt/best/encoder.pt ✓


## ⚙️ Configuration
Adjust these before running.

In [9]:
from config import Phase2Config

cfg = Phase2Config(
    # Data
    # shuffle_buffer=50000, # This will pull the whole dataset into your 50GB RAM

    streaming=False,

    max_samples=None,            # Train on the full dataset
    batch_size=4,                # Keep batch small to fit T5 + CTC on GPU
    gradient_accumulation_steps=4, # Effective batch size = 16

    # Architecture - Max Implementation
    use_attention_pooling=True,  # Smart temporal aggregation
    use_ctc_head=True,           # Auxiliary temporal alignment loss
    ctc_weight=0.1,              # Balanced weight for CTC loss

    # Training Schedule
    epochs=10,
    warmup_epochs=2,             # 2 epochs of frozen encoder to protect Phase 1 weights
    lr_encoder=5e-6,             # Very slow fine-tuning for encoder
    lr_adapter=1e-4,             # Standard rate for new weights
    lr_t5=5e-5,                  # Gentle fine-tuning for T5

    # Performance & Reliability
    mixed_precision=True,        # Faster training
    num_beams=4,                 # Better generation quality at validation time

    # Infrastructure
    phase1_ckpt="/content/phase1_ckpt", # Point to your new Phase 1
    ckpt_dir="/content/phase2_ckpt",
    upload_hf=True,
    hf_repo="mycyrilgoud/data255",
    seed=15179996,
)

print(f"Config: {cfg}")

Config: Phase2Config(dataset_repo='bdanko/how2sign-landmarks-front-raw-parquet', split='train', max_samples=100, val_max_samples=20, batch_size=8, max_target_length=128, d_model=384, latent_dim=512, encoder_layers=3, encoder_heads=8, encoder_dropout=0.1, use_part_embeddings=True, t5_name='google/flan-t5-small', t5_dim=512, adapter_dropout=0.1, use_attention_pooling=True, pool_num_heads=4, epochs=3, warmup_epochs=1, lr_encoder=5e-06, lr_adapter=0.0001, lr_t5=5e-05, weight_decay=1e-05, grad_clip=1.0, scheduler='cosine', warmup_steps=100, mixed_precision=False, gradient_accumulation_steps=1, num_beams=4, max_new_tokens=50, use_ctc_head=False, ctc_weight=0.1, ctc_vocab_size=256, ckpt_dir='/content/phase2_ckpt', phase1_ckpt='/content/phase1_ckpt', hf_repo='mycyrilgoud/data255', upload_hf=True, seed=42, log_backend='csv', wandb_project='cslt-phase2', log_every_n_steps=10, smoke_test=False, run_id='7ba1c9e3')


## Training
Runs the Phase 2 fine-tuning loop with utterance-level supervision, staged unfreezing, and full evaluation (BLEU, ROUGE-L, chrF, exact match).

In [7]:
from phase2_finetune import train_phase2

train_phase2(cfg)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

Val E3: 0it [00:00, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]


Sample Predictions
  [1] Pred: economy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy
       Ref:  I'll show you.

  [2] Pred: foreign energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy
       Ref:  So that is one way, you can just let them go right by.

  [3] Pred: heating energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy energy ene

## Upload to Hugging Face (optional)

In [10]:
from huggingface_hub import HfApi, create_repo, upload_folder

if cfg.upload_hf:
    create_repo(cfg.hf_repo, exist_ok=True)
    upload_folder(
        folder_path=cfg.ckpt_dir,
        repo_id=cfg.hf_repo,
        commit_message=f"Phase 2 checkpoint — {cfg.epochs} epochs",
    )
    print(f"Uploaded to https://huggingface.co/{cfg.hf_repo} ✓")
else:
    print("Skipping upload. Set cfg.upload_hf = True to upload.")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...e2_ckpt/best/optimizer.pt:   0%|          | 8.88kB /  684MB            

  ..._ckpt/latest/optimizer.pt:   0%|          | 8.88kB /  684MB            

  ...ase2_ckpt/best/adapter.pt:   1%|1         | 30.8kB / 2.11MB            

...

  ...e2_ckpt/latest/adapter.pt:   1%|1         | 30.8kB / 2.11MB            

  ...ase2_ckpt/latest/model.pt:   0%|          | 28.0kB /  350MB            

  ...phase2_ckpt/best/model.pt:   1%|1         | 3.97MB /  350MB            

  ...e2_ckpt/latest/encoder.pt:   8%|8         | 2.87MB / 35.5MB            

Uploaded to https://huggingface.co/mycyrilgoud/data255 ✓


## Quick Inference Test

In [11]:
from config import InferenceConfig
from inference import run_inference

inf_cfg = InferenceConfig(
    ckpt_dir=cfg.ckpt_dir,
    num_samples=3,
)
run_inference(inf_cfg)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]


CSLT Inference — Sign Language → English Translation

[Sample 1]
  Video ID:    cwLJfXFf9ks_7-8-rgb_front
  Frames:      67
  Reference:   I'll show you.
  Prediction:  foreign the environment the health the environment the health the environment the environment the environment the environment the health the environment the environment the environment the environment the environment the environment the environment the environment the environment the environment the environment the environment the environment the environment the people the

[Sample 2]
  Video ID:    cwLJfXFf9ks_8-8-rgb_front
  Frames:      72
  Reference:   So that is one way, you can just let them go right by.
  Prediction:  economy the economy the economy the environment the environment the environment the environment the environment the environment the environment the environment the environment the environment the environment the environment the environment the environment the environment the environment the enviro